# Zambia National ETD Portal — metadata filling: a documented walkthrough

**Project:** [Zambia National ETD Portal](https://datalab.unza.zm/projects/zambia-national-etd-portal) (https://netd.ac.zm), DataLab Research Group ([https://datalab.unza.zm](https://datalab.unza.zm)).
**Requirements covered:** SRS v0.25 FR-14 and FR-48 to FR-53, NFR-14; design in SDD v0.23 §4.1.1, §5.2 and §5.3; operations in `docs/OPERATIONS.md`.
**Prepared:** November 1, 2026.

This notebook documents, step by step and with runnable code, the machinery the portal uses to fill metadata that institutional repositories leave empty: degree level, degree, discipline, supervisor, department and faculty. Every step calls the production code in `harvester/netd/ml/`, so what you see here is what runs on the server.

**It never touches the live server.** It reads public records from the portal's REST API and public PDFs from the repositories, and works in its own throwaway SQLite store (`netd-notebook-work/demo.sqlite3`).

| § | Step | Code | Requirement |
|---|---|---|---|
| 1 | Where this comes from: the earlier DataLab work | — | — |
| 2 | Architecture and the order of precedence | — | FR-14 |
| 3 | Setup | — | — |
| 4 | Sample records from the REST API, as canonical records | `netd.store` | EI-6 |
| 5 | PDF text cache (front matter first; OCR for scans) | `netd/ml/textcache.py` | FR-48 |
| 6 | The baseline model, rule by rule | `netd/ml/models/titlepage.py` | FR-49 |
| 7 | The prediction cache and input hashing | `netd/ml/predict.py`, `cache.py` | FR-49 |
| 8 | The overlay: thresholds, provenance, "derived" | `netd/ml/overlay.py` | FR-14, FR-50 |
| 9 | Degree level: automatic exclusion and the review list | `overlay.py`, `cli.exclusion` | FR-47, FR-52 |
| 10 | Manual corrections | `cache.py`, admin panel | FR-51 |
| 11 | Predictions computed elsewhere (Colab, LLMs) | `cache.load_batch` | FR-49 |
| 12 | What the portal and the API show | `netd/index.py`, `api_record.rb` | FR-50 |
| 13 | Evaluation: gold-standard sample and metrics | `netd/ml/evaluate.py` | FR-53 |
| 14 | Writing and registering a new model | model contract | FR-49 |
| 15 | Running it in production | `netd.cli` | all |
| 16 | Next steps towards the paper | — | — |
| 17 | References | — | — |

## 1. Where this comes from: the earlier DataLab work

The portal's metadata filling continues a line of DataLab studies. The table shows what each contributed and how this implementation uses it.

| Work | What it did | Result | Used here |
|---|---|---|---|
| Phiri (2020), *IJMSO* 14(3) | Supervised classification of UNZA ETDs: **ETD type** (Master's/Doctoral) from cover-page text (CountVectorizer + Random Forest), and **collection** (school) from title + abstract (TF-IDF + SGD); released as the `etd_autoclassifier` Flask API | Type: weighted F1 0.99 (684 ETDs; Doctoral recall 0.74). Collection: weighted F1 0.81 over 11 schools | Same target fields (degree level, faculty); background only. The next models use current methods instead (§16); the 2020 pickles no longer load in current scikit-learn in any case |
| Chipangila et al. (2021, *JCDL*; 2023, *IJDL*) | Controlled vocabularies and multi-label subject classification (arXiv subjects) for UNZA's repository | 71.2% of items had ≤ 2 subject headings; 92.1% of headings used once; SUS 68.9 vs 66.2 with controlled vocabularies | Subject is the next field to fill (not in this baseline) |
| Kasonde, Chisale & Phiri (2022); Kasonde & Phiri (2023); Chisale & Phiri (2023); Chisale (2024) | ETD-MS compliance, metadata-quality policy, and the sources of missing ETD metadata in Zambian repositories | Supervisor, department, degree level largely absent from harvested records | Defines the gaps (all sources: supervisor 0%, department 0% on November 1, 2026) and the preliminary pages as sources |
| Chanda, Kanfwa & Mainda (2023), ICT 4014 report | Zambia National ETD pre-processing pipeline | — | The harvest → clean → enrich pipeline this plugs into |
| Nyambe, Muzumala, Mulenga, Mufuzi, Ngoma & Phiri (unpublished manuscript, in revision) | Supervisor (`dc.contributor.advisor`) extraction from the **Acknowledgements page**: spaCy NER rules vs. GPT-3.5; ground truth of **3,266 annotated UNZA ETDs** (title page, copyright declaration, certificate of approval, acknowledgements) | Accuracy 66% (rules) vs 70% (LLM) on 832 observations | Confirms the acknowledgements as an evidence source, alongside the title and approval pages (§6); its annotated ETDs are a candidate gold standard for §13 |

**What is new here** is the operational layer that turns these one-off experiments into a running service: a cache of PDF text with OCR, a prediction cache keyed by input so models only rerun when input changes, explicit provenance and confidence on every value, a confidence threshold for display, "derived" marking for users and API clients, manual corrections that win over everything, a review list for degree levels that would remove records, and an evaluation harness that compares any model with the baseline on the same gold standard.

## 2. Architecture and the order of precedence

```
 records (harvested, cleaned)
      │ pdf_url, files
      ▼
 textcache.extract-text / ocr-text ──► pdf_text  (first pages, front matter first)
                                          │
      harvested values only ──────────────┤
                                          ▼
 models (titlepage-rules, …) ──► predict ──► predictions  (value, confidence, evidence, input hash)
 Colab / LLM batches ──► load-predictions ─┘
 admin panel ──► netd.cli correct ──► corrections
                                          │
            netd.cli process: crosswalk → enrichers → overlay.apply → exclusion (FR-47) → records → Solr
                                                          └──► level_review (admin panel)
```

For every field, the value shown on the portal is decided in this order:

1. **Manual correction** (admin panel) — always wins, survives re-harvests (FR-51).
2. **Harvested value** from the repository — never overwritten by a model.
3. **Predicted value** — only if the field is empty **and** confidence ≥ `show_threshold` (0.8); marked *derived* (FR-14, FR-50).

Degree levels below postgraduate are special because they remove a record from the portal: they are applied only at ≥ 0.95, and between 0.6 and 0.95 they go to a review list instead (FR-52).

## 3. Setup

The code lives in the private repository `lightonphiri/code-datalab26-netd_aggregator`. Run this notebook either

* **locally** from `notebooks/` in a clone of the repository (the next cell finds `../harvester`), or
* **in Google Colab**: the next cell asks for a GitHub personal access token with read access and clones the repository into `/content/netd`. The token is not printed or stored in the notebook.

Set `N_UNZA`, `N_ZCAS` and `N_MU` to change the sample size, and `RUN_OCR = True` to try OCR (Colab installs Tesseract with `apt`; it takes about 20 seconds per scanned PDF).

In [1]:
import os, sys, json, pathlib, subprocess, random, copy, inspect, time, warnings
warnings.simplefilter("ignore", SyntaxWarning)

N_UNZA, N_ZCAS, N_MU = 10, 3, 2   # sample size per institution
RUN_OCR = False                    # True: OCR scanned PDFs (needs tesseract-ocr and poppler-utils)
SEED = 2026

REPO = pathlib.Path(os.environ.get("NETD_REPO", "..")).resolve()
if not (REPO / "harvester" / "netd").exists():          # not inside a clone: Colab
    from getpass import getpass
    token = getpass("GitHub token (read access to lightonphiri/code-datalab26-netd_aggregator): ")
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    f"https://{token}@github.com/lightonphiri/code-datalab26-netd_aggregator.git", "/content/netd"],
                   check=True)
    del token
    REPO = pathlib.Path("/content/netd")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "harvester" / "requirements.txt"), "pandas"],
               check=True)
if RUN_OCR and "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y tesseract-ocr tesseract-ocr-eng poppler-utils > /dev/null", shell=True)
sys.path.insert(0, str(REPO / "harvester"))

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from netd.store import Store
from netd import cli, index
from netd.ml import cache, textcache, predict, evaluate
from netd.ml.overlay import Overlay
from netd.ml.models import titlepage

WORK = pathlib.Path("netd-notebook-work"); WORK.mkdir(exist_ok=True)
DB = WORK / "demo.sqlite3"
for suffix in ("", "-wal", "-shm"):
    pathlib.Path(str(DB) + suffix).unlink(missing_ok=True)
store = Store(str(DB))      # same class the server uses: WAL mode, 120 s busy timeout (NFR-14)
cache.init(store)
store.db.executescript(textcache.SCHEMA)
print("repository:", REPO)
print("model:", titlepage.NAME, "version", titlepage.VERSION)
print("tables:", [r[0] for r in store.db.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

repository: /home/claude/netd
model: titlepage-rules version 4
tables: ['bitstreams', 'corrections', 'excluded', 'harvest_runs', 'level_review', 'pdf_text', 'predictions', 'raw_records', 'records', 'set_names', 'sqlite_sequence']


The staging store gains four tables for metadata filling (SRS §3.4):

| Table | Key | Holds |
|---|---|---|
| `pdf_text` | record | files read, status (`ok`, `ocr`, `no_text`, `ocr_failed`, `too_large`, `error`), pages, text of the first pages, text hash |
| `predictions` | record, field, model | model version, value list, confidence, input hash, evidence |
| `corrections` | record, field | value list (`[]` clears), note, administrator, time; field `_scope` = include/exclude decision |
| `level_review` | record | predicted level, confidence, model, evidence, status (`pending`/`decided`) |

## 4. Sample records from the REST API, as canonical records

On the server, records come from OAI-PMH harvesting and the crosswalk. Here we take public records from the portal's REST API (`/api/v1/records`) and turn them back into the canonical form the harvester stores (SDD §4.2).

One rule matters already: **values the API marks as `derived` were produced by a model, so they are dropped**. Models must only ever see harvested values (FR-49), otherwise a model would learn from its own output.

In [2]:
import urllib.request, urllib.parse
API = "https://netd.ac.zm/api/v1"

def api(path, **params):
    url = f"{API}/{path}?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "NETD notebook"})
    for attempt in range(4):                    # the API is rate-limited per address: back off and retry
        try:
            return json.load(urllib.request.urlopen(req, timeout=60))
        except OSError:
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))

API_TO_CANONICAL = {"title": "title", "authors": "author", "supervisors": "supervisor", "subjects": "subject",
                    "degree": "degree", "degree_level": "degree_level", "discipline": "discipline",
                    "faculty": "faculty", "department": "department", "year": "year"}

def to_canonical(a):
    derived = set(a.get("derived") or {})          # model output: not evidence
    rec = {"id": a["id"], "source": a["institution"]["id"], "institution": a["institution"]["name"],
           "institution_short": a["institution"]["id"].upper(), "oai_id": a["oai_identifier"], "provenance": {}}
    for af, cf in API_TO_CANONICAL.items():
        v = a.get(af)
        if af in derived or v in (None, "", []):
            continue
        rec[cf] = [str(x) for x in v] if isinstance(v, list) else [str(v)]
        rec["provenance"][cf] = "harvested:api"
    rec["files"] = [{"url": f["url"], "name": f.get("name"), "mime": f.get("mime_type"), "size": f.get("size")}
                    for f in a.get("files") or []]
    pdfs = [f["url"] for f in rec["files"] if "pdf" in (f["mime"] or "")] or ([a["links"]["pdf"]] if a["links"].get("pdf") else [])
    if pdfs:
        rec["pdf_url"] = pdfs
    return rec

rng = random.Random(SEED)
sample = []
for inst, n in (("unza", N_UNZA), ("zcas", N_ZCAS), ("mu", N_MU)):
    total = api("records", institution=inst, full_text="true", per_page=1)["meta"]["total"]
    n_pages = max(1, -(-total // 20))                      # pages of 20, rounded up
    pages = sorted(rng.sample(range(1, n_pages + 1), min(4, n_pages)))
    pool = [r for p in pages for r in api("records", institution=inst, full_text="true", per_page=20, sort="title", page=p)["data"]]
    sample += rng.sample(pool, min(n, len(pool)))

records = [to_canonical(a) for a in sample]
for r in records:
    store.put_record(r)
store.commit()
pd.DataFrame([{"id": r["id"], "title": r["title"][0][:70], "harvested level": (r.get("degree_level") or [""])[0],
               "supervisor": len(r.get("supervisor") or []), "department": len(r.get("department") or []),
               "PDF files": len(r.get("pdf_url") or [])} for r in records])

,id,title,harvested level,supervisor,department,PDF files
0,unza-123456789-6137,Assessment of selected properties of mine tailings at the former Kabwe,,0,0,1
1,unza-123456789-7581,Investigating the perception and challenges of the existence of tribal,,0,0,1
2,unza-123456789-7897,The role of social capital in the employability of university graduate,,0,0,1
3,unza-123456789-4972,The role of School Managers in Mitigating High School Drop out rate of,Masters,0,0,1
4,unza-123456789-7922,The relationship between emotional intelligence and effective leadersh,,0,0,1
5,unza-123456789-5013,The role of Sex Education in mitigating teenage pregnancy in selected,Masters,0,0,1
6,unza-123456789-7573,The role of school management in effecting learner discipline in publi,,0,0,1
7,unza-123456789-477,The Role of professional Associations in continuing professional devel,,0,0,2
8,unza-123456789-5603,The relationship between general self-efficacy and adaptation among fi,,0,0,1
9,unza-123456789-4213,Investigating the role of media in communicating messages on the elimi,,0,0,1


The gaps are visible at once: supervisor and department are empty for every record, and many have no degree level. These are the fields the machinery fills.

## 5. PDF text cache (FR-48)

Evidence for the missing fields is in the **preliminary pages** of each thesis: the title page ("submitted in partial fulfilment of the requirements for the award of the degree of Master of …"), the certificate of approval and declaration (supervisor), and the acknowledgements ("my supervisor, Dr …"), as the Nyambe et al. study (in revision) found.

### 5.1 Which files to read: front matter first

Some repositories split a thesis into several files. For many UNZA records the first PDF is "Main Document.pdf", which starts at Chapter 1, while "Cover page.pdf" and "Preliminary pages.pdf" hold the evidence. `front_urls()` puts cover/title files first, then other front-matter names, then the rest (at most 3 files); with no front-matter file, only the first PDF is read.

In [3]:
print(inspect.getsource(textcache.front_urls))
multi = [r for r in records if len(r.get("pdf_url") or []) > 1]
for r in multi[:3]:
    names = {f["url"]: f["name"] for f in r["files"]}
    print(r["id"], "\n   repository order:", [names.get(u) for u in r["pdf_url"]],
          "\n   read order:      ", [names.get(u) for u in textcache.front_urls(r)])

def front_urls(rec, most=3):
    """The PDFs to read for a record, front matter first. Some repositories split a thesis into several files
    ("Cover page.pdf", "Preliminary pages.pdf", "Main Document.pdf"); the cover and preliminary pages carry the
    degree statement and supervisors, so they are read before the main document."""
    pdfs = list(rec.get("pdf_url") or [])
    if len(pdfs) < 2:
        return pdfs
    names = {f.get("url"): f.get("name") or "" for f in rec.get("files") or []}
    rank = lambda u: (0 if re.search(r"cover|title", names.get(u, ""), re.I) else
                      1 if FRONT.search(names.get(u, "")) else 2)
    ordered = sorted(pdfs, key=lambda u: (rank(u), pdfs.index(u)))
    if rank(ordered[0]) == 2:
        return ordered[:1]  # no front-matter file: the first PDF holds the whole thesis
    return ordered[:most]

unza-123456789-477 
   repository order: ['NDOPU0001.PDF', 'NdopuNG20001.PDF'] 
   read order:       ['NDOPU0001.PDF']


### 5.2 Extracting the first pages

`extract-text` downloads each file (at most 60 MB), keeps the text of the first 8 pages (`pypdf`) and stores one row per record. Pages are separated by form feeds (`\f`), files by a form feed too. A PDF whose text layer has fewer than 80 characters is a scan: status `no_text`, to be OCRed.

On the server this runs in batches of 500 under the harvest lock, with two download threads; SQLite is written only from the main thread. Here we run it on the sample.

In [4]:
textcache.run(store, pages=8, workers=2, pause=0.5)
rows = store.db.execute("SELECT record_id, status, pages, length(text) chars, message FROM pdf_text").fetchall()
pd.DataFrame([dict(r) for r in rows])

extract-text: 15 PDFs to fetch


extract-text: finished 15 (12 with text)


,record_id,status,pages,chars,message
0,unza-123456789-6137,ok,61.0,9710.0,NaN
1,unza-123456789-7581,ok,76.0,7550.0,NaN
2,unza-123456789-7897,ok,19.0,22777.0,NaN
3,unza-123456789-4972,ok,72.0,4427.0,NaN
4,unza-123456789-7922,ok,89.0,7585.0,NaN
5,unza-123456789-5013,no_text,75.0,NaN,no text layer (scanned?)
6,unza-123456789-7573,ok,89.0,13573.0,NaN
7,unza-123456789-477,no_text,84.0,NaN,no text layer (scanned?)
8,unza-123456789-5603,ok,135.0,11261.0,NaN
9,unza-123456789-4213,ok,221.0,7941.0,NaN


### 5.3 OCR for scanned PDFs

Most UNZA theses deposited before about 2015 are scans (one font, hundreds of images). `ocr-text` handles the `no_text` rows one at a time: `pdftoppm` renders the first 5 pages at 200 dpi in greyscale, Tesseract (English) reads them, both at the lowest CPU priority (`nice 19`), because the server has 2 CPUs shared with other services. Status becomes `ocr` or `ocr_failed`.

In [5]:
import shutil
if RUN_OCR and shutil.which("tesseract") and shutil.which("pdftoppm"):
    textcache.run_ocr(store, pages=5)
else:
    print("OCR skipped (set RUN_OCR = True; needs tesseract-ocr and poppler-utils).")
textcache.summary(store)

OCR skipped (set RUN_OCR = True; needs tesseract-ocr and poppler-utils).
mu       error           1
mu       ok              1
unza     no_text         2
unza     ok              8
zcas     ok              3


## 6. The baseline model, rule by rule (FR-49)

`titlepage-rules` is deliberately transparent: every value comes with the text it was read from, and a confidence that reflects how specific that evidence is. It is the baseline every later model is compared with (§13).

| Field | Evidence | Confidence |
|---|---|---|
| degree, discipline, degree level | degree statement: *fulfilment / requirement / submitted / award* followed within 200 characters by *Master / Doctor / Bachelor / Postgraduate Diploma of/in …* | 0.96 (discipline 0.91) |
| | a degree name anywhere on the first three pages | 0.82 |
| | an abbreviation (MSc, PhD, PGD, BSc …), case-sensitive | 0.70 (level only) |
| supervisor | "Supervisor: Dr X" | 0.92 |
| | "supervised by X" / "under the supervision of X" | 0.90 / 0.88 |
| | acknowledgements: "my supervisor(s), Dr X and Prof Y" / "Dr X, my supervisor" | 0.86 / 0.82 |
| | "X (Supervisor)" | 0.75 |
| department / faculty | a title-page line that *starts* with "Department of …" / "School of …" / "Faculty of …" (≤ 80 characters) | 0.88 on page 1, 0.78 later |

Two problems of real PDF text are handled explicitly:

* **Words split by text extraction** ("M aster", "Zambi a", "Eco nomics"). Degree keywords are matched letter by letter with optional spaces (`fz()`); other words are rejoined when the joined word is more frequent in the whole corpus than the rarer fragment (`prepare()` counts words across all cached texts).
* **Run-on text.** Units are read line by line, so the thesis title on the next line does not run into "Department of Public Health"; names are cut at words such as *Date*, *Signature*, *Head*, and must end in a surname of at least 4 letters.

In [6]:
texts = {r["record_id"]: r["text"] for r in store.db.execute("SELECT record_id, text FROM pdf_text WHERE status IN ('ok','ocr')")}
titlepage.prepare(texts.values())          # corpus word counts, used to rejoin split words
demo = "THE UNIVERSITY OF ZAMBIA\nSCHOOL OF EDUCATION\nDepartment of Library and Information Science\n" \
       "A dissertation submitted to the University of Zambi a in partial fulfilment of the requirements for the " \
       "award of the degree of M aster of Library and Information Science (MLIS)\nLusaka 2019" \
       "\fACKNOWLEDGEMENTS\nI sincerely appreciate my academic supervisor, Dr. Lighton Phiri, whose careful review ..."
for field, (values, conf, evidence) in titlepage.predict({}, demo).items():
    print(f"{field:13} {values}  confidence {conf:.2f}\n{'':13} evidence: …{evidence[:110]}…")

degree        ['Master of Library and Information Science']  confidence 0.96
              evidence: …epartment of Library and Information Science A dissertation submitted to the University of Zambi a in partial …
degree_level  ['Masters']  confidence 0.96
              evidence: …epartment of Library and Information Science A dissertation submitted to the University of Zambi a in partial …
supervisor    ['Lighton Phiri']  confidence 0.86
              evidence: …incerely appreciate my academic supervisor, Dr. Lighton Phiri, whose careful revi…
department    ['Department of Library and Information Science']  confidence 0.88
              evidence: …Department of Library and Information Science…
faculty       ['School of Education']  confidence 0.88
              evidence: …SCHOOL OF EDUCATION…


The same model on the real sample (first record with text), showing each value with its evidence:

In [7]:
rid = next(iter(texts))
out = titlepage.predict({}, texts[rid])
print(rid)
pd.DataFrame([{"field": f, "value": " || ".join(v), "confidence": c, "evidence": (e or "")[:140]} for f, (v, c, e) in out.items()])

unza-123456789-6137


,field,value,confidence,evidence
0,degree,Master of Science in Integrated Soil Fertility Management,0.96,IAL TO SUPPORT PLANT GROWTH By Mercy Mulenga A dissertation submitted in partial fulfilment of the requirement s for...
1,degree_level,Masters,0.96,IAL TO SUPPORT PLANT GROWTH By Mercy Mulenga A dissertation submitted in partial fulfilment of the requirement s for...
2,discipline,Integrated Soil Fertility Management,0.91,IAL TO SUPPORT PLANT GROWTH By Mercy Mulenga A dissertation submitted in partial fulfilment of the requirement s for...


## 7. The prediction cache and input hashing (FR-49)

`predict.run()` gives each model, for each record, the PDF text and **only the harvested values** of the fields the model declares in `USES` (values whose provenance starts with `model:` or `corrected:` are removed). It hashes that input together with the model version, and calls the model only if the hash changed since the last run. Fields a model covers but finds nothing for are stored as `[]`, so they are not recomputed either.

So the nightly run costs almost nothing when nothing changed, and bumping `VERSION` re-runs a model over everything.

In [8]:
print(inspect.getsource(predict.harvested_view))
predict.run(store)          # first run: every record with text
predict.run(store)          # second run: nothing changed, nothing recomputed
pd.read_sql_query("SELECT record_id, field, value, round(confidence,2) confidence, model||'-'||version model, "
                  "substr(evidence,1,90) evidence FROM predictions WHERE value != '[]' ORDER BY record_id, field", store.db)

def harvested_view(rec, fields):
    """The record's harvested values for `fields` (derived and corrected values removed)."""
    prov = rec.get("provenance", {})
    return {f: rec.get(f) for f in fields
            if rec.get(f) and not str(prov.get(f, "")).startswith(("model:", "corrected:"))}

predict titlepage-rules-4: 12 records with input, 12 (re)predicted


predict titlepage-rules-4: 12 records with input, 0 (re)predicted


,record_id,field,value,confidence,model,evidence
0,mu-123456789-139,degree,"[""Master of Disaster Studies""]",0.96,titlepage-rules-4,TUENCY OF NDOLA URBAN MARY LESA 201601582 A Research Report submitted in fulfillment of th
1,mu-123456789-139,degree_level,"[""Masters""]",0.96,titlepage-rules-4,TUENCY OF NDOLA URBAN MARY LESA 201601582 A Research Report submitted in fulfillment of th
2,mu-123456789-139,faculty,"[""School of Agriculture and Natural Resources Disaster Management Training Center""]",0.78,titlepage-rules-4,School of Agriculture and Natural Resources Disaster Management Training Center
3,mu-123456789-139,supervisor,"[""Mwansa Kamukwamba""]",0.92,titlepage-rules-4,lugushi University. Supervisor: Dr. Mwansa Kamukwamba Signature…………..... ..........
4,unza-123456789-4213,degree,"[""Master of Communication for Development""]",0.96,titlepage-rules-4,", NK WAZI AND TWAPIA TOWNSHIPS. By MWAMBA SEFELINO A report submitted to the Uni versity o"
5,unza-123456789-4213,degree_level,"[""Masters""]",0.96,titlepage-rules-4,", NK WAZI AND TWAPIA TOWNSHIPS. By MWAMBA SEFELINO A report submitted to the Uni versity o"
6,unza-123456789-4972,degree,"[""Master of Education in Educational Management""]",0.96,titlepage-rules-4,UDY OF MUNALI SPECIAL UNIT BY: Malama Yvonne A dissertation Submitted to the University of
7,unza-123456789-4972,degree_level,"[""Masters""]",0.96,titlepage-rules-4,UDY OF MUNALI SPECIAL UNIT BY: Malama Yvonne A dissertation Submitted to the University of
8,unza-123456789-4972,discipline,"[""Educational Management""]",0.91,titlepage-rules-4,UDY OF MUNALI SPECIAL UNIT BY: Malama Yvonne A dissertation Submitted to the University of
9,unza-123456789-4972,supervisor,"[""Austin Cheyeka""]",0.90,titlepage-rules-4,ational Management. SUPERVISED BY: PROF. AUSTIN. CHEYEKA UNZA/ZOU 2017 DECLARATIO


## 8. The overlay: thresholds, provenance and "derived" (FR-14, FR-50)

During `netd.cli process`, after the crosswalk and the built-in enrichers and **before** the postgraduate-scope check (FR-47), `Overlay.apply()` merges the caches into each record:

* a record with neither predictions nor corrections is returned **byte-identical** (so its OAI-PMH datestamp does not move);
* an empty field takes the best prediction if its confidence ≥ `show_threshold` (0.8), with `provenance[field] = "model:<name>-<version>"`, `confidence[field]` and `derived[field]`;
* corrections are applied last and overwrite everything (§10).

The configuration lives in `harvester/enrichers.yml` under `ml:`.

In [9]:
import yaml
ecfg = yaml.safe_load(open(REPO / "harvester" / "enrichers.yml"))
print("ml:", ecfg["ml"], "\nmin_confidence (facets):", ecfg["min_confidence"])
ov = Overlay(store, dict(ecfg["ml"], min_confidence=ecfg["min_confidence"]))

def show_overlay(rid):
    before = next(r for r in store.records() if r["id"] == rid)
    after, scope = ov.apply(copy.deepcopy(before))
    rows = []
    for f in ("degree_level", "degree", "discipline", "supervisor", "department", "faculty"):
        rows.append({"field": f, "before": " || ".join(before.get(f) or []), "after": " || ".join(after.get(f) or []),
                     "provenance": after.get("provenance", {}).get(f, ""),
                     "derived": json.dumps(after.get("derived", {}).get(f)) if f in after.get("derived", {}) else ""})
    return pd.DataFrame(rows), after

with_preds = [r[0] for r in store.db.execute("SELECT DISTINCT record_id FROM predictions WHERE value != '[]'")]
table, after = show_overlay(with_preds[0])
print(with_preds[0]); table

ml: {'show_threshold': 0.8, 'level_auto_exclude': 0.95, 'level_review_min': 0.6, 'nightly': True, 'nightly_pdfs': 300, 'nightly_ocr': 30} 
min_confidence (facets): 0.6
mu-123456789-139


,field,before,after,provenance,derived
0,degree_level,,Masters,model:titlepage-rules-4,"{""method"": ""model:titlepage-rules-4"", ""confidence"": 0.96}"
1,degree,,Master of Disaster Studies,model:titlepage-rules-4,"{""method"": ""model:titlepage-rules-4"", ""confidence"": 0.96}"
2,discipline,,,,
3,supervisor,,Mwansa Kamukwamba,model:titlepage-rules-4,"{""method"": ""model:titlepage-rules-4"", ""confidence"": 0.92}"
4,department,,,,
5,faculty,,,,


## 9. Degree level: automatic exclusion and the review list (FR-47, FR-52)

The portal holds postgraduate work only (FR-47). A prediction that a thesis is **below** postgraduate level would remove it, so it needs stronger evidence:

| Predicted below-postgraduate level, confidence | Effect |
|---|---|
| ≥ 0.95 (`level_auto_exclude`) | level applied; FR-47 excludes the record, the reason names the model |
| 0.6 – 0.95 (`level_review_min`) | record stays in the portal; listed under **Admin → Level review** with evidence and PDF link |
| < 0.6 | ignored |

An administrator's decision is stored as a `_scope` correction (`include` or `exclude`) and overrides both the model and FR-47's level rule. We simulate both cases on two sample records (these two predictions are invented for the demonstration and deleted afterwards).

In [10]:
# two throwaway copies of a sample record, without a degree level and without predictions of their own
base = copy.deepcopy(records[0])
for k in ("degree_level", "degree", "discipline"):
    base.pop(k, None); base["provenance"].pop(k, None)
a, b = "demo-auto-exclude", "demo-review"
for rid in (a, b):
    store.put_record(dict(copy.deepcopy(base), id=rid))
cache.put_prediction(store, a, "degree_level", "demo", "0", ["Bachelors"], 0.97, "demo", "…for the award of the degree of Bachelor of Arts…")
cache.put_prediction(store, b, "degree_level", "demo", "0", ["Bachelors"], 0.80, "demo", "…B.Sc …")
store.commit()

def decide(note):
    ov = Overlay(store, dict(ecfg["ml"], min_confidence=ecfg["min_confidence"]))
    for rid in (a, b):
        rec, scope = ov.apply(next(r for r in store.records() if r["id"] == rid))
        print(f"{note:18} {rid:18} level={rec.get('degree_level')}  exclusion: {cli.exclusion(rec, {'id': rec['source']}, {}, scope)}")
    print(f"{note:18} review list: {[(r[0], r[1], r[2]) for r in ov.review]}\n")

decide("model only:")
cache.put_correction(store, b, "_scope", ["include"], "admin@example.org", "checked the PDF: Master's dissertation")
decide("after the decision:")

for rid in (a, b):          # remove the demonstration
    store.delete_record(rid); cache.delete_correction(store, rid, "_scope")
store.db.execute("DELETE FROM predictions WHERE model='demo'"); store.commit()

model only:        demo-auto-exclude  level=['Bachelors']  exclusion: degree level Bachelors is out of scope (only Doctoral, Masters, Postgraduate Diploma from model:demo-0)
model only:        demo-review        level=None  exclusion: None
model only:        review list: [('demo-review', 'Bachelors', 0.8)]

after the decision: demo-auto-exclude  level=['Bachelors']  exclusion: degree level Bachelors is out of scope (only Doctoral, Masters, Postgraduate Diploma from model:demo-0)
after the decision: demo-review        level=None  exclusion: None
after the decision: review list: []



## 10. Manual corrections (FR-51)

Corrections are entered in the admin panel (**Admin → Correct**: record id or portal address). The panel opens the store read-only and writes through `netd.cli correct`, the same function used below; every correction is logged in the activity log with the administrator and note. A correction:

* overrides harvested **and** predicted values (`provenance = corrected:<e-mail>`), or clears a field with an empty value;
* survives re-harvests, because it lives in its own table and is applied at every processing run;
* takes effect when the source is next processed (nightly, or **Apply now**, which runs `netd.cli reprocess --source <id>`);
* is a labelled example for training and evaluating the next model (§13).

In [11]:
rid = with_preds[0]
cache.put_correction(store, rid, "supervisor", ["Lighton Phiri"], "lighton.phiri@unza.ac.zm", "demonstration")
store.commit()
ov = Overlay(store, dict(ecfg["ml"], min_confidence=ecfg["min_confidence"]))
rec, _ = ov.apply(next(r for r in store.records() if r["id"] == rid))
print("supervisor:", rec["supervisor"], "| provenance:", rec["provenance"]["supervisor"], "| derived:", "supervisor" in rec.get("derived", {}))
cache.delete_correction(store, rid, "supervisor"); store.commit()      # remove the demonstration

supervisor: ['Lighton Phiri'] | provenance: corrected:lighton.phiri@unza.ac.zm | derived: False


## 11. Predictions computed elsewhere: Colab notebooks and LLMs (FR-49)

Heavy models (transformers, spaCy `en_core_web_trf`, LLM calls as in the Nyambe et al. study (in revision)) should not run on the 3.8 GB server. They run elsewhere and their output is loaded into the same cache as **JSON Lines**, one prediction per line:

```json
{"record_id": "unza-123456789-1000", "field": "supervisor", "model": "ack-llm", "version": "1",
 "value": ["Henry J. Msango"], "confidence": 0.9, "evidence": "…my supervisor, Mr Henry J. Msango, for…"}
```

On the server: `netd.cli load-predictions FILE.jsonl`, then `netd.cli reprocess`. The overlay then picks, per field, the prediction with the highest confidence among all models. Below, a batch is written from the baseline's own supervisor predictions purely to show the format and the loading step.

In [12]:
batch = WORK / "example-batch.jsonl"
with open(batch, "w") as fh:
    for r in store.db.execute("SELECT * FROM predictions WHERE field='supervisor' AND value != '[]' LIMIT 3"):
        fh.write(json.dumps({"record_id": r["record_id"], "field": "supervisor", "model": "example-batch", "version": "0",
                             "value": json.loads(r["value"]), "confidence": round(r["confidence"] - 0.05, 2),
                             "evidence": r["evidence"]}) + "\n")
print(batch.read_text()[:400])
cache.load_batch(store, str(batch))
store.db.execute("DELETE FROM predictions WHERE model='example-batch'"); store.commit()

{"record_id": "unza-123456789-4972", "field": "supervisor", "model": "example-batch", "version": "0", "value": ["Austin Cheyeka"], "confidence": 0.85, "evidence": "ational Management. SUPERVISED BY: PROF. AUSTIN. CHEYEKA UNZA/ZOU 2017 DECLARATIO"}
{"record_id": "unza-123456789-7922", "field": "supervisor", "model": "example-batch", "version": "0", "value": ["F. Hambulo"], "confidence": 0.81, "evid
loaded 3 predictions from netd-notebook-work/example-batch.jsonl


## 12. What the portal and the API show (FR-50)

Derived values travel to Solr as `derived_ssm` (one JSON object per field). The record page puts a small **derived** badge next to each such value (tooltip: model and confidence, "may be wrong; use Report a problem") and a note under *More About This Work*. The REST API record (API 1.1.0) gains a `derived` object — an addition only, so existing clients are unaffected (EI-6).

In [13]:
doc = index.to_solr(after)
print("Solr derived_ssm:", json.dumps(doc.get("derived_ssm"), indent=1))
api_names = {"supervisor": "supervisors", "subject": "subjects", "author": "authors"}
print("API 'derived':", json.dumps({api_names.get(d["field"], d["field"]): {"method": d["method"], "confidence": d["confidence"]}
                                   for d in map(json.loads, doc.get("derived_ssm", []))}, indent=1))
live = api(f"records/{after['id']}")
print("\nlive API record today has 'derived':", json.dumps(live.get("derived")))

Solr derived_ssm: [
 "{\"method\": \"model:titlepage-rules-4\", \"confidence\": 0.96, \"field\": \"degree\"}",
 "{\"method\": \"model:titlepage-rules-4\", \"confidence\": 0.96, \"field\": \"degree_level\"}",
 "{\"method\": \"model:titlepage-rules-4\", \"confidence\": 0.92, \"field\": \"supervisor\"}"
]
API 'derived': {
 "degree": {
  "method": "model:titlepage-rules-4",
  "confidence": 0.96
 },
 "degree_level": {
  "method": "model:titlepage-rules-4",
  "confidence": 0.96
 },
 "supervisors": {
  "method": "model:titlepage-rules-4",
  "confidence": 0.92
 }
}



live API record today has 'derived': null


## 13. Evaluation: gold-standard sample and metrics (FR-53)

No model is enabled on the portal until it has been measured against a gold standard, and every later model is compared with this baseline on the same records.

* `netd.cli gold-sample gold.csv --n 300` draws a sample stratified by institution (proportional, at least 20 each) and writes the harvested values plus empty `gold_*` columns. Annotators fill them from the PDF (several values separated by `||`; `-` when the PDF does not state the field).
* `netd.cli evaluate gold.csv` prints, per model, version and field, **coverage** (share of gold records the model gave a value for), **precision**, **recall** and **F1** at confidence thresholds 0, 0.8 and 0.95. Values are normalised before comparison (case, accents, titles such as Dr/Prof, unit words); people are compared by surname and first initial, so "Dr L. Phiri" matches "Phiri, Lighton".
* The **3,266 UNZA ETDs annotated for the Nyambe et al. study (in revision)** cover the same preliminary pages and are the natural gold standard for UNZA; corrections made in the admin panel add labels from every institution over time.

Below: the sample file, then a *silver-standard* check that needs no annotation: for records whose repository **does** state a degree level, compare the model's level prediction (made from the PDF only; the model never sees the harvested level) with the repository's value.

In [14]:
evaluate.sample(store, str(WORK / "gold-sample.csv"), n=10, seed=SEED)
print(pd.read_csv(WORK / "gold-sample.csv").iloc[:3, :8].to_string())

silver = WORK / "silver-degree_level.csv"
labelled = [r for r in records if r.get("degree_level")]
pd.DataFrame([{"record_id": r["id"], "gold_degree_level": r["degree_level"][0]} for r in labelled]).to_csv(silver, index=False)
print(f"\nsilver standard: {len(labelled)} sample records with a repository degree level")
evaluate.evaluate(store, str(silver))

wrote 15 records to netd-notebook-work/gold-sample.csv (mu 2, unza 10, zcas 3)
             record_id institution                                                                                                                                                                              title                                      portal_url                                                                                             pdf_url harvested_degree_level harvested_degree harvested_discipline
0  unza-123456789-7922        UNZA  The relationship between emotional intelligence and effective leadership of female head teachers in management of human resource in selected secondary schools of Lusaka district  https://netd.ac.zm/catalog/unza-123456789-7922     https://rims.unza.ac.zm/server/api/core/bitstreams/96dc3d76-4428-4f76-a393-f6d71cf3c02b/content                    NaN              NaN                  NaN
1  unza-123456789-7573        UNZA                                      The

With a sample this small the figures only show that the harness works. The production evaluation runs on the server over the full text cache, against the annotated gold standard.

## 14. Writing and registering a new model (FR-49)

A model is a Python module with this contract (SDD §5):

| Name | Meaning |
|---|---|
| `NAME`, `VERSION` | identify the model; bump `VERSION` to recompute everything |
| `FIELDS` | the fields it predicts |
| `USES` | harvested record fields it reads (e.g. `("title", "abstract")`) |
| `NEEDS_TEXT` | `False` for models that only use metadata (e.g. the 2020 title + abstract collection classifier) |
| `prepare(texts)` | optional, once per run with all cached texts (vocabularies, loading a pickled classifier) |
| `predict(meta, text)` | returns `{field: (values, confidence, evidence)}` |

Register it in `MODELS` in `netd/ml/predict.py` and it runs nightly. The example below is a minimal model in the spirit of Algorithms 1 and 2 of Nyambe et al. (in revision) (find the Acknowledgements page, keep sentences containing "supervi", take capitalised names), registered at run time for the demonstration.

In [15]:
model_src = '''
import re
NAME, VERSION = "ack-sentences", "0"
FIELDS, USES, NEEDS_TEXT = ("supervisor",), (), True
NAME_RX = re.compile(r"(?:Dr|Prof(?:essor)?|Mr|Mrs|Ms)\.?\\s+((?:[A-Z][a-zA-Z'-]+|[A-Z]\\.)(?:\\s+(?:[A-Z][a-zA-Z'-]+|[A-Z]\\.)){1,3})")

def predict(meta, text):
    pages = text.split("\\f")
    # Algorithm 1: the page whose heading is "Acknowledgements" (the table of contents also mentions the word,
    # but not at the top of the page)
    ack = next((p for p in pages if re.search(r"acknowledge?ments?", p.strip()[:120], re.I)), "")
    names, evidence = [], None
    for sentence in re.split(r"(?<=[.!?])\\s+", " ".join(ack.split())):                 # Algorithm 2
        if "supervi" in sentence.lower():
            found = NAME_RX.findall(sentence)
            names += [n for n in found if n not in names]
            evidence = evidence or sentence[:200]
    return {"supervisor": (names, 0.7, evidence)} if names else {}
'''
(WORK / "ack_sentences.py").write_text(model_src)
sys.path.insert(0, str(WORK))
predict.MODELS["ack-sentences"] = "ack_sentences"
predict.run(store, model="ack-sentences")
pd.read_sql_query("SELECT record_id, model, value, confidence FROM predictions WHERE field='supervisor' "
                  "ORDER BY record_id, model", store.db)          # [] = the model found nothing

predict ack-sentences-0: 12 records with input, 12 (re)predicted


,record_id,model,value,confidence
0,mu-123456789-139,ack-sentences,"[""Mwansa Kamukwamba""]",0.70
1,mu-123456789-139,titlepage-rules,"[""Mwansa Kamukwamba""]",0.92
2,unza-123456789-4213,ack-sentences,[],NaN
3,unza-123456789-4213,titlepage-rules,[],NaN
4,unza-123456789-4972,ack-sentences,[],NaN
5,unza-123456789-4972,titlepage-rules,"[""Austin Cheyeka""]",0.90
6,unza-123456789-5603,ack-sentences,[],NaN
7,unza-123456789-5603,titlepage-rules,"[""Sophie Kasonde""]",0.86
8,unza-123456789-6137,ack-sentences,[],NaN
9,unza-123456789-6137,titlepage-rules,[],NaN


As expected, this minimal acknowledgements-only model finds far fewer supervisors than the baseline, which also reads labelled supervisor lines on the title and approval pages and handles multiple supervisors and split words. Both sets of predictions sit side by side in the cache; the overlay shows the most confident one, and `evaluate` (§13) measures each against the same gold standard. That is how every new model is judged before it is enabled.

## 15. Running it in production

All commands run on the server in `/opt/netd/harvester` as user `netd` (`sudo -u netd .venv/bin/python -m netd.cli …`); full runbook in `docs/OPERATIONS.md`.

| Task | Command |
|---|---|
| PDF text for new or changed records | `flock -w 7200 run.lock … extract-text --limit 500` (`--retry-failed` for repository errors) |
| OCR scanned PDFs | `flock -w 7200 run.lock … ocr-text --limit 50` |
| Whole backlog, surviving disconnects | `systemd-run --unit=netd-mltext --uid=netd …` (see OPERATIONS.md); follow with `journalctl -u netd-mltext -f` |
| Status | `text-report`; `export-predictions /opt/netd/logs/predictions.csv` |
| Run models | `predict` (changed input only), `predict --all` |
| Load external predictions | `load-predictions FILE.jsonl` |
| Apply to the portal | dry run first on a copy (`sqlite3 netd.sqlite3 ".backup /opt/netd/logs/dry.sqlite3"`, `process --db …`), then `reprocess` |
| Corrections from the shell | `correct <id> <field> <value…> --by <e-mail> --note …`; `uncorrect <id> <field>` |
| Gold standard, evaluation | `gold-sample gold.csv --n 300`; `evaluate gold.csv` |

**Nightly.** The scheduled `netd.cli run` does, per source: harvest → text for up to 300 new or changed PDFs → up to 30 OCRs → `predict` → process (overlay, FR-47) → index. A failure in metadata filling is reported and skipped; it never stops the harvest (`ml.nightly`, `nightly_pdfs`, `nightly_ocr` in `enrichers.yml`).

**Safety.** The store is in SQLite WAL mode so the admin panel, reports and backups can read while a writer works (NFR-14); open it only as `netd` (never as root), and keep long jobs under `run.lock` in batches so the nightly harvest waits at most one batch.

## 16. Next steps: modern models for the paper

The rules model is the transparent baseline. The next models use current methods, all within the contracts above (models on the server run in-process; anything needing a GPU runs elsewhere and is loaded as a batch, §11).

1. **Layout-aware reading of the preliminary pages.** Replace plain `pypdf` text and Tesseract with a document-conversion pipeline that keeps layout (e.g. Docling or Marker), and a vision-language OCR model for the ~1,100 scanned UNZA PDFs (e.g. olmOCR or a small Qwen-VL-class model), run on a GPU. Classify each page (title, declaration, approval, acknowledgements, abstract, contents) and store the page type with the text, so every model reads the right page.
2. **One grounded, schema-constrained extraction pass.** A small open-weight instruction LLM (7–14B, quantised) extracts the whole ETD-MS record at once (degree name and level, discipline, supervisors with roles, department, faculty, grantor) under a JSON schema enforced by constrained decoding (e.g. vLLM guided decoding or Outlines). Every value must cite a verbatim evidence span; values whose span is not found in the page text are discarded, which removes hallucinated names.
3. **Calibrated confidence and selective prediction.** Replace hand-set confidences with calibrated ones: features such as self-consistency across samples, token probabilities, agreement with the rules model and evidence location, mapped to probabilities by isotonic regression on the gold standard. Display thresholds are then chosen for a target precision (e.g. 95%) from risk–coverage curves, optionally with conformal guarantees.
4. **Distillation to a CPU model.** Once the LLM is validated, use it to label every ETD, then fine-tune a small encoder (e.g. ModernBERT or DeBERTa-v3, or GLiNER for entities) that runs nightly on the server's CPU, so new ETDs are filled the same night without GPU or API costs.
5. **Normalisation against authority lists.** Fill controlled values, not free text: resolve supervisor name variants into one person per institution (blocking, string and embedding similarity, institutional constraints; ORCID where it exists); map departments and schools to each HEI's organisational units; map degrees to the HEA-accredited programme list; map disciplines to ISCED-F 2013 fields. This is what makes facets work.
6. **Embeddings for subjects, discipline, duplicates and search.** Scholarly sentence embeddings (e.g. SPECTER2 or E5/BGE-class models) of title and abstract support k-nearest-neighbour or zero-shot classification into a controlled vocabulary, duplicate detection (FR-19) and, later, semantic search.
7. **Human in the loop.** Admin corrections and review decisions become training and evaluation data; annotation effort goes where models disagree or are least certain (active learning), with inter-annotator agreement reported for the gold standard.

**Compute:** GPU work (VLM OCR, LLM extraction, fine-tuning) is a one-off backfill plus occasional retraining on free notebook GPUs (Kaggle, Colab), returned as batches; the nightly pipeline on the server stays CPU-only (rules + distilled model). See `docs/PAPER-PLAN.md`.

**Evaluation for the paper:** field-level precision, recall and F1 with normalised matching; risk–coverage curves and calibration error; generalisation across institutions (train on UNZA, test on ZCAS, Mulungushi and UNILUS); born-digital versus scanned; ablations (front matter first, page classification, grounding check, distillation); and cost, time and energy per 1,000 ETDs on the hardware a national aggregator in the Global South actually has.


## 17. References

* Phiri, L. (2020). Automatic classification of digital objects for improved metadata quality of electronic theses and dissertations in institutional repositories. *International Journal of Metadata, Semantics and Ontologies*, 14(3), 234–248. https://doi.org/10.1504/IJMSO.2020.112804
* Phiri, L. (2019). *etd_autoclassifier* v1.0 (ETD type and collection classifiers). https://github.com/lightonphiri/etd_autoclassifier/tree/v1.0
* Chipangila, B., Liswaniso, E., Mawila, A., Mwanza, P., Nawila, D., M'sendo, R., Nyirenda, M. & Phiri, L. (2021). Improved discoverability of digital objects in institutional repositories using controlled vocabularies. *2021 ACM/IEEE Joint Conference on Digital Libraries (JCDL)*, 100–109. https://doi.org/10.1109/JCDL52503.2021.00022
* Chipangila, B., Liswaniso, E., Mawila, A., Mwanza, P., Nawila, D., M'sendo, R., Nyirenda, M. & Phiri, L. (2023). Controlled vocabularies in digital libraries: challenges and solutions for increased discoverability of digital objects. *International Journal on Digital Libraries*, 25(2), 139–155. https://doi.org/10.1007/s00799-023-00374-1
* Nyambe, F., Muzumala, M., Mulenga, N., Mufuzi, R., Ngoma, G. & Phiri, L. (in revision). *Automated metadata completeness: automatic generation of missing electronic theses and dissertations metadata*. Unpublished manuscript, DataLab Research Group, University of Zambia; preprint to be posted on arXiv.
* Kasonde, C., Chisale, A. & Phiri, L. (2022). Empirical evaluation of ETD-MS compliance in institutional repositories. *ETD 2022*, Novi Sad. https://doi.org/10.5281/zenodo.7084847
* Kasonde, C. & Phiri, L. (2023). Assessing and promoting metadata quality for electronic theses and dissertations in institutional repositories using a policy-driven approach. *ETD 2023*, Gujarat. http://ir.inflibnet.ac.in/handle/1944/2412
* Chisale, A. & Phiri, L. (2023). Towards metadata completeness in national ETD portals for improved discoverability. *ETD 2023*, Gujarat. http://ir.inflibnet.ac.in/handle/1944/2446
* Chisale, A. (2024). *Identification of sources for missing electronic theses and dissertations metadata in higher education in Zambia* (MLIS dissertation). University of Zambia. https://datalab.unza.zm/sites/default/files/2024-06/etds-datalab24-mlis-dissertation-adrian_chisale.pdf
* Chanda, C., Kanfwa, C. & Mainda, C. (2023). *Zambia National ETD pre-processing pipeline* (ICT 4014 project report). University of Zambia. https://datalab.unza.zm/sites/default/files/2023-12/reports-unza23-ict4014-chanda_kanfwa_mainda.pdf